# Google Analytics Campaign Alignment

Builds two outputs:
- `monthly_gmv_per_ga_view`: monthly portfolio-level GMV per GA view
- `data_output/combined_restaurant_ga.parquet`: restaurant-month momentum data enriched with GA view metrics


In [ ]:
from pathlib import Path

import pandas as pd

BASE_DIR = Path("..")
RESTAURANTS_PATH = BASE_DIR / "_2_feature_engineering+momentum" / "data_output" / "restaurants_agg_performance.parquet"
GMV_VIEW_PATH = BASE_DIR / "_5_GA_data" / "data_output" / "gmv" / "gmv_view.parquet"
OUTPUT_DIR = Path("data_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMBINED_OUT = OUTPUT_DIR / "combined_restaurant_ga.parquet"

res = pd.read_parquet(RESTAURANTS_PATH)
gmv_view_df = pd.read_parquet(GMV_VIEW_PATH)

def _safe_divide(numerator, denominator):
    return pd.to_numeric(numerator, errors="coerce") / pd.to_numeric(denominator, errors="coerce").replace(0, pd.NA)


## Validate Restaurant-Month Grain


In [ ]:
gmv_view = gmv_view_df.copy()
gmv_view["restaurant_id"] = pd.to_numeric(gmv_view["restaurant_id"], errors="coerce").astype("Int64")
gmv_view["year_month"] = pd.to_datetime(gmv_view["year_month"], errors="coerce").dt.to_period("M").dt.to_timestamp()

duplicate_keys = int(gmv_view.duplicated(["restaurant_id", "year_month"]).sum())
print("gmv_view rows:", len(gmv_view))
print("duplicate restaurant-month rows:", duplicate_keys)
print("restaurants with GA data:", gmv_view["restaurant_id"].nunique())
print("months with GA data:", gmv_view["year_month"].nunique())

if duplicate_keys:
    gmv_view.loc[
        gmv_view.duplicated(["restaurant_id", "year_month"], keep=False),
        ["restaurant_id", "name", "year_month", "ga_items_viewed", "monthly_gmv"],
    ].sort_values(["restaurant_id", "year_month"]).head(20)
else:
    gmv_view[["restaurant_id", "name", "year_month", "ga_items_viewed", "monthly_gmv", "gmv_per_view"]].head()


## Monthly GMV per GA View


In [ ]:
restaurant_ga_monthly = (
    gmv_view_df.copy()
    .assign(
        restaurant_id=lambda df: pd.to_numeric(df["restaurant_id"], errors="coerce").astype("Int64"),
        year_month=lambda df: pd.to_datetime(df["year_month"], errors="coerce").dt.to_period("M").dt.to_timestamp(),
    )
    .sort_values(["restaurant_id", "year_month"])
    .reset_index(drop=True)
)
restaurant_ga_monthly["gmv_per_ga_view"] = restaurant_ga_monthly["gmv_per_view"]
restaurant_ga_monthly["bookings_per_ga_view"] = _safe_divide(
    restaurant_ga_monthly["monthly_bookings"],
    restaurant_ga_monthly["ga_items_viewed"],
)
restaurant_ga_monthly["ga_view_to_purchase_rate"] = restaurant_ga_monthly["view_to_purchase_rate"]

monthly_gmv_per_ga_view = (
    restaurant_ga_monthly
    .groupby("year_month", as_index=False)
    .agg(
        restaurants_with_ga_data=("restaurant_id", "nunique"),
        total_monthly_gmv=("monthly_gmv", "sum"),
        total_monthly_bookings=("monthly_bookings", "sum"),
        total_ga_items_viewed=("ga_items_viewed", "sum"),
        total_ga_items_added_to_cart=("ga_items_added_to_cart", "sum"),
        total_ga_items_purchased=("ga_items_purchased", "sum"),
        total_ga_item_revenue=("ga_item_revenue", "sum"),
    )
    .sort_values("year_month")
)

monthly_gmv_per_ga_view["gmv_per_ga_view"] = _safe_divide(monthly_gmv_per_ga_view["total_monthly_gmv"], monthly_gmv_per_ga_view["total_ga_items_viewed"])
monthly_gmv_per_ga_view["bookings_per_ga_view"] = _safe_divide(monthly_gmv_per_ga_view["total_monthly_bookings"], monthly_gmv_per_ga_view["total_ga_items_viewed"])
monthly_gmv_per_ga_view["ga_view_to_purchase_rate"] = _safe_divide(monthly_gmv_per_ga_view["total_ga_items_purchased"], monthly_gmv_per_ga_view["total_ga_items_viewed"])

monthly_gmv_per_ga_view


## Restaurant-Month Combined Output


In [ ]:
restaurant_perf = res.copy()
restaurant_perf["restaurant_id"] = pd.to_numeric(restaurant_perf["restaurant_id"], errors="coerce").astype("Int64")
restaurant_perf["year_month"] = pd.to_datetime(restaurant_perf["year_month"], errors="coerce").dt.to_period("M").dt.to_timestamp()

ga_monthly = (
    gmv_view_df[
        [
            "restaurant_id",
            "year_month",
            "ga_items_viewed",
            "ga_items_added_to_cart",
            "ga_items_purchased",
            "ga_item_revenue",
            "gmv_per_view",
            "bookings_per_view",
            "ga_add_to_cart_rate",
            "view_to_purchase_rate",
            "purchase_to_cart_rate",
            "revenue_per_view",
        ]
    ]
    .copy()
    .assign(
        restaurant_id=lambda df: pd.to_numeric(df["restaurant_id"], errors="coerce").astype("Int64"),
        year_month=lambda df: pd.to_datetime(df["year_month"], errors="coerce").dt.to_period("M").dt.to_timestamp(),
        gmv_per_ga_view=lambda df: df["gmv_per_view"],
        bookings_per_ga_view=lambda df: df["bookings_per_view"],
        ga_view_to_purchase_rate=lambda df: df["view_to_purchase_rate"],
        ga_purchase_to_cart_rate=lambda df: df["purchase_to_cart_rate"],
        ga_revenue_per_view=lambda df: df["revenue_per_view"],
    )
)

duplicate_keys = int(ga_monthly.duplicated(["restaurant_id", "year_month"]).sum())
if duplicate_keys:
    raise ValueError(f"gmv_view.parquet is not unique at restaurant-month grain: {duplicate_keys} duplicate keys")

combined = restaurant_perf.merge(
    ga_monthly,
    on=["restaurant_id", "year_month"],
    how="left",
    validate="one_to_one",
)

combined.to_parquet(COMBINED_OUT, index=False)
print(f"Saved: {COMBINED_OUT}")
print(combined.shape)
print("rows with GA data:", int(combined["ga_items_viewed"].fillna(0).gt(0).sum()))
combined.head()
